<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/LlamaIndex_RAG_simple_single_company.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LlamaIndex RAG Demo: Document Q&A with a Framework

This notebook demonstrates how **LlamaIndex** simplifies building RAG pipelines. Instead of manually chunking documents, computing embeddings, and querying vector stores, LlamaIndex handles the plumbing so you can focus on the application logic.

**How it works:**
1. Create a small set of **synthetic company documents** (no external data needed).
2. Use LlamaIndex to **ingest, chunk, embed, and index** the documents in one step.
3. Given a natural-language question, LlamaIndex **retrieves** relevant chunks and sends them to an LLM.
4. Compare answers **with RAG** (LlamaIndex pipeline) vs. **without RAG** (direct LLM call).

**Learning goals:**
- Understand what a RAG framework (LlamaIndex) does under the hood
- See how LlamaIndex abstracts document loading, chunking, embedding, and retrieval
- Compare the framework approach to the manual RAG we built in earlier notebooks
- Observe how RAG grounding eliminates hallucination on private/synthetic data

**Provider setup:** This notebook supports **7 free-tier LLM APIs** with automatic fallback:
Gemini → Ollama → Grok (xAI) → Groq → HuggingFace → Cohere → OpenRouter.

Store whichever API keys you have in Colab Secrets (or a local `.env` file):
`GEMINI_API_KEY`, `OLLAMA_API_KEY`, `XAI_API_KEY`, `GROQ_API_KEY`, `HF_TOKEN`, `COHERE_API_KEY`, `OPENROUTER_API_KEY`

In [1]:
!pip install -q -U llama-index llama-index-llms-openai-like llama-index-llms-gemini llama-index-embeddings-huggingface google-genai openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16

## 1) Imports and Provider Helpers (7-Vendor Cascade)

We configure **LlamaIndex** to use whichever LLM provider you have available.
The cascade tries each provider in order and falls through on quota errors.

For **embeddings**, we use a local HuggingFace model (`all-MiniLM-L6-v2`) so that
embedding always works regardless of which LLM API key you have.

In [2]:
import os
from pathlib import Path

# ---- .env loader (for local VS Code / laptop testing) -----------------------
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
if env_path.exists():
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

# ---- Colab Secrets ----------------------------------------------------------
try:
    from google.colab import userdata  # type: ignore
except Exception:
    userdata = None


def _colab_secret(name):
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None


def _get_key(env_name):
    """Look up a key from env vars first, then Colab Secrets."""
    return os.getenv(env_name) or _colab_secret(env_name)


# ---- Provider definitions (in fallback order) --------------------------------
#
# | #  | Provider     | Free Tier                          | Signup URL                             |
# |----|--------------|------------------------------------|----------------------------------------|
# | 1  | Gemini       | 500 req/day (Flash)                | https://aistudio.google.com/apikey     |
# | 2  | Ollama Cloud | Free tier                          | https://ollama.com                     |
# | 3  | Grok (xAI)   | $25/month free credits             | https://console.x.ai                  |
# | 4  | Groq         | 30 req/min, 14400 req/day          | https://console.groq.com              |
# | 5  | HuggingFace  | Free serverless inference          | https://huggingface.co/settings/tokens |
# | 6  | Cohere       | 20 req/min free (trial key)        | https://dashboard.cohere.com/api-keys  |
# | 7  | OpenRouter   | Free models available (rotating)   | https://openrouter.ai/keys             |
#
PROVIDERS = [
    {
        "name": "Gemini",
        "key_env": "GEMINI_API_KEY",
        "style": "gemini",
        "default_model": "gemini-2.5-flash",
    },
    {
        "name": "Ollama",
        "key_env": "OLLAMA_API_KEY",
        "style": "openai",
        "base_url": "https://ollama.com/v1",
        "default_model": "kimi-k2.5:cloud",
    },
    {
        "name": "Grok (xAI)",
        "key_env": "XAI_API_KEY",
        "style": "openai",
        "base_url": "https://api.x.ai/v1",
        "default_model": "grok-3-mini",
    },
    {
        "name": "Groq",
        "key_env": "GROQ_API_KEY",
        "style": "openai",
        "base_url": "https://api.groq.com/openai/v1",
        "default_model": "llama-3.3-70b-versatile",
    },
    {
        "name": "HuggingFace",
        "key_env": "HF_TOKEN",
        "style": "openai",
        "base_url": "https://router.huggingface.co/v1",
        "default_model": "meta-llama/Llama-3.3-70B-Instruct",
    },
    {
        "name": "Cohere",
        "key_env": "COHERE_API_KEY",
        "style": "openai",
        "base_url": "https://api.cohere.ai/compatibility/v1",
        "default_model": "command-r-plus",
    },
    {
        "name": "OpenRouter",
        "key_env": "OPENROUTER_API_KEY",
        "style": "openai",
        "base_url": "https://openrouter.ai/api/v1",
        "default_model": "meta-llama/llama-3.3-70b-instruct:free",
    },
]


def _is_quota_error(exc):
    msg = str(exc).lower()
    return any(s in msg for s in (
        "quota", "resource_exhausted", "429", "rate limit",
        "exceeded your current quota", "too many requests",
        "rate_limit_exceeded", "tokens per minute",
    ))


def get_available_providers():
    return [p for p in PROVIDERS if _get_key(p["key_env"])]


def has_llm_provider():
    return len(get_available_providers()) > 0


# ---- Print status -----------------------------------------------------------
available = get_available_providers()
if available:
    print("Providers configured (in fallback order):")
    for p in available:
        print(f"  + {p['name']:<16} model = {p['default_model']}")
    missing = [p for p in PROVIDERS if p not in available]
    if missing:
        print("Not configured (skipped):")
        for p in missing:
            print(f"  - {p['name']:<16} (set {p['key_env']})")
else:
    print("WARNING: No API keys found. Set at least one of:")
    for p in PROVIDERS:
        print(f"  {p['key_env']}")

Providers configured (in fallback order):
  + OpenRouter       model = meta-llama/llama-3.3-70b-instruct:free
Not configured (skipped):
  - Gemini           (set GEMINI_API_KEY)
  - Ollama           (set OLLAMA_API_KEY)
  - Grok (xAI)       (set XAI_API_KEY)
  - Groq             (set GROQ_API_KEY)
  - HuggingFace      (set HF_TOKEN)
  - Cohere           (set COHERE_API_KEY)


## 2) Configure LlamaIndex LLM and Embeddings

LlamaIndex needs two things:
- An **LLM** for generating answers (we pick the first available provider from our cascade)
- An **embedding model** for converting text to vectors (we use a local HuggingFace model — no API key needed)

This cell configures LlamaIndex's global `Settings` so all downstream components use our chosen models.

In [3]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ---- Embedding model (local, no API key needed) -----------------------------
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedding model: sentence-transformers/all-MiniLM-L6-v2 (local)")

# ---- LLM (first available provider from cascade) ----------------------------
llm_provider = None

for provider in get_available_providers():
    api_key = _get_key(provider["key_env"])
    try:
        if provider["style"] == "gemini":
            from llama_index.llms.gemini import Gemini
            Settings.llm = Gemini(api_key=api_key, model="models/" + provider["default_model"])
        else:
            from llama_index.llms.openai_like import OpenAILike
            Settings.llm = OpenAILike(
                api_key=api_key,
                api_base=provider["base_url"],
                model=provider["default_model"],
                is_chat_model=True,
            )
        llm_provider = provider
        print(f"LLM provider: {provider['name']} ({provider['default_model']})")
        break
    except Exception as e:
        print(f"  Skipping {provider['name']}: {e}")
        continue

if llm_provider is None:
    print("ERROR: Could not configure any LLM provider. Check your API keys.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2 (local)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_7964/770333681.py:16: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  Settings.llm = Gemini(api_key=api_key, model="models/" + provider["default_model"])


LLM provider: Gemini (gemini-2.5-flash)


## 3) Create Synthetic Company Documents

We generate a small set of **fictional company documents** that the LLM has never seen in training. This ensures that correct answers can **only** come from RAG retrieval, not from parametric memory.

| Document | Topic |
|---|---|
| Company Overview | Acme Analytics Inc. — founding, mission, HQ |
| Products & Services | Three product lines with pricing |
| Q4 2025 Financial Results | Revenue, profit, growth numbers |
| Employee Handbook (excerpt) | PTO policy, remote work rules |
| Customer Case Study | BigRetail Corp deployment results |

In [4]:
DOCUMENTS = {
    "company_overview.txt": """ACME ANALYTICS INC. — COMPANY OVERVIEW

Acme Analytics Inc. was founded in 2019 by Dr. Sarah Chen and Marcus Rivera
in Lawrence, Kansas. The company specializes in AI-powered business intelligence
tools for mid-market companies (100–5,000 employees).

Headquarters: 1420 Jayhawk Boulevard, Lawrence, KS 66045
Employees: 287 (as of January 2026)
Annual Revenue (2025): $42.3 million
Funding: Series B ($18M raised in March 2023 from Midwest Ventures)

Mission: "To democratize data analytics so every business decision is informed
by evidence, not intuition."

The company operates three offices: Lawrence (HQ), Chicago (sales), and
Austin (engineering). CEO: Dr. Sarah Chen. CTO: Marcus Rivera. CFO: Linda Park.""",

    "products_and_services.txt": """ACME ANALYTICS — PRODUCTS AND SERVICES

1. InsightBoard Pro (Flagship Product)
   - Real-time dashboard platform with natural language query interface
   - Pricing: $49/user/month (Standard), $89/user/month (Enterprise)
   - Supports PostgreSQL, MySQL, Snowflake, BigQuery, and Redshift
   - 1,847 active enterprise customers as of Q4 2025

2. DataPipe ETL
   - Automated data pipeline builder with 200+ pre-built connectors
   - Pricing: starts at $500/month for up to 10 million records/day
   - Launched in September 2024

3. PredictIQ
   - ML-powered forecasting add-on for InsightBoard Pro
   - Pricing: $29/user/month (requires InsightBoard Pro subscription)
   - Uses proprietary time-series model trained on retail and logistics data
   - Beta launched March 2025, GA release planned for June 2026

All products include 24/7 email support. Enterprise plans include dedicated
account manager and 99.9% SLA.""",

    "q4_2025_financials.txt": """ACME ANALYTICS — Q4 2025 FINANCIAL RESULTS (CONFIDENTIAL)

Period: October 1 – December 31, 2025

Revenue:          $12.8 million (Q4) / $42.3 million (FY 2025)
Gross Margin:     78.2%
Operating Profit: $1.9 million (Q4) / $5.1 million (FY 2025)
Net Income:       $1.4 million (Q4)
Cash on Hand:     $14.7 million
Burn Rate:        Company is cash-flow positive since Q2 2025

Key Metrics:
- ARR (Annual Recurring Revenue): $48.6 million (up 34% YoY)
- Net Revenue Retention: 118%
- Customer Acquisition Cost (CAC): $8,200
- Customer Lifetime Value (LTV): $67,400
- LTV/CAC Ratio: 8.2x

Headcount grew from 241 to 287 employees during 2025.
R&D spending: 31% of revenue. Sales & Marketing: 28% of revenue.""",

    "employee_handbook_excerpt.txt": """ACME ANALYTICS — EMPLOYEE HANDBOOK (EXCERPT)

PAID TIME OFF (PTO):
- All full-time employees receive 22 days of PTO per year (accrued monthly).
- PTO increases to 27 days after 3 years of service.
- Unused PTO can be carried over (max 5 days) or paid out at year-end.
- Sick leave: 10 days per year (separate from PTO).

REMOTE WORK POLICY:
- Engineering and Data Science teams: fully remote eligible.
- Sales and Customer Success: hybrid (minimum 2 days/week in office).
- All employees may work remotely up to 4 weeks/year from any US location.
- International remote work requires VP approval and tax review.

PROFESSIONAL DEVELOPMENT:
- Annual learning budget: $2,500 per employee.
- Conference attendance: up to 2 conferences per year with manager approval.
- Tuition reimbursement: up to $5,250/year for degree programs.""",

    "customer_case_study.txt": """CUSTOMER CASE STUDY: BIGRETAIL CORP

Company: BigRetail Corp (1,200 retail stores across 38 states)
Challenge: Siloed data across POS, inventory, and CRM systems made it
impossible for regional managers to get timely insights.

Solution: Deployed InsightBoard Pro Enterprise + DataPipe ETL
- Connected 14 data sources in 3 weeks using DataPipe
- 340 regional managers now use InsightBoard daily
- Natural language queries replaced manual SQL report requests

Results (after 6 months):
- 62% reduction in time-to-insight (from 4 days to 1.5 days average)
- $3.2 million saved in inventory carrying costs
- 23% increase in regional manager satisfaction scores
- SQL report request backlog eliminated entirely

Quote from BigRetail CIO Janet Torres:
\"InsightBoard Pro transformed how our managers interact with data.
They went from waiting days for a report to asking questions in plain
English and getting answers in seconds.\"""",
}

# Write documents to disk
DOC_DIR = Path("acme_docs")
DOC_DIR.mkdir(exist_ok=True)

for filename, content in DOCUMENTS.items():
    (DOC_DIR / filename).write_text(content, encoding="utf-8")

print(f"Created {len(DOCUMENTS)} documents in '{DOC_DIR}/':")
for f in sorted(DOC_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size} bytes)")

Created 5 documents in 'acme_docs/':
  company_overview.txt (715 bytes)
  customer_case_study.txt (924 bytes)
  employee_handbook_excerpt.txt (828 bytes)
  products_and_services.txt (915 bytes)
  q4_2025_financials.txt (709 bytes)


## 4) Ingest Documents into LlamaIndex

This is where LlamaIndex shines — **one line** to load all documents, and **one line** to build a searchable index.

Under the hood, LlamaIndex:
1. Reads each `.txt` file
2. Splits text into chunks (default ~1024 tokens with overlap)
3. Computes embeddings for each chunk using our HuggingFace model
4. Stores them in an in-memory vector index

In [5]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Step 1: Load documents from the folder
documents = SimpleDirectoryReader(str(DOC_DIR)).load_data()
print(f"Loaded {len(documents)} document chunks")

# Step 2: Build vector index (embeds + indexes all chunks)
index = VectorStoreIndex.from_documents(documents, show_progress=True)
print(f"Vector index built with {len(documents)} chunks")

# Step 3: Create a query engine
query_engine = index.as_query_engine(similarity_top_k=3)
print("Query engine ready (top_k=3 retrieval)")

Loaded 5 document chunks


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

Vector index built with 5 chunks
Query engine ready (top_k=3 retrieval)


## 5) What Just Happened? Peeking Under the Hood

Let's inspect what LlamaIndex created — how many chunks, what the embeddings look like, and how retrieval works before we query the LLM.

In [6]:
# Inspect the chunks LlamaIndex created
retriever = index.as_retriever(similarity_top_k=3)

# Test retrieval without LLM — just see what chunks come back
test_query = "How much revenue did Acme make in Q4 2025?"
retrieved_nodes = retriever.retrieve(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_nodes)} chunks:\n")
for i, node in enumerate(retrieved_nodes, 1):
    source = node.metadata.get('file_name', 'unknown')
    score = node.score
    text_preview = node.text[:200].replace('\n', ' ')
    print(f"  Chunk #{i} (score={score:.4f}, source={source})")
    print(f"    {text_preview}...\n")

Query: 'How much revenue did Acme make in Q4 2025?'
Retrieved 3 chunks:

  Chunk #1 (score=0.6737, source=q4_2025_financials.txt)
    ACME ANALYTICS — Q4 2025 FINANCIAL RESULTS (CONFIDENTIAL)  Period: October 1 – December 31, 2025  Revenue:          $12.8 million (Q4) / $42.3 million (FY 2025) Gross Margin:     78.2% Operating Profi...

  Chunk #2 (score=0.4163, source=company_overview.txt)
    ACME ANALYTICS INC. — COMPANY OVERVIEW  Acme Analytics Inc. was founded in 2019 by Dr. Sarah Chen and Marcus Rivera in Lawrence, Kansas. The company specializes in AI-powered business intelligence too...

  Chunk #3 (score=0.3867, source=employee_handbook_excerpt.txt)
    ACME ANALYTICS — EMPLOYEE HANDBOOK (EXCERPT)  PAID TIME OFF (PTO): - All full-time employees receive 22 days of PTO per year (accrued monthly). - PTO increases to 27 days after 3 years of service. - U...



## 6) Query with RAG vs. Without RAG

Now we compare:
- **With RAG (LlamaIndex):** The query engine retrieves relevant chunks, sends them + the question to the LLM, and returns a grounded answer.
- **Without RAG (Direct LLM):** We ask the same question directly to the LLM with no context. Since the data is fictional, the LLM either refuses or hallucinates.

For the "without RAG" call we use `generate_text()` from our 7-vendor cascade.

In [7]:
from google import genai
from openai import OpenAI


def generate_text_no_rag(prompt, system_prompt=None):
    """Direct LLM call via the 7-vendor cascade — no RAG context."""
    available = get_available_providers()
    if not available:
        raise ValueError("No LLM provider configured.")

    last_exc = None
    for provider in available:
        api_key = _get_key(provider["key_env"])
        try:
            if provider["style"] == "gemini":
                client = genai.Client(api_key=api_key)
                contents = prompt if not system_prompt else f"{system_prompt}\n\n{prompt}"
                resp = client.models.generate_content(
                    model=provider["default_model"], contents=contents,
                )
                return resp.text, provider["name"]
            else:
                client = OpenAI(api_key=api_key, base_url=provider["base_url"])
                messages = []
                if system_prompt:
                    messages.append({"role": "system", "content": system_prompt})
                messages.append({"role": "user", "content": prompt})
                resp = client.chat.completions.create(
                    model=provider["default_model"], messages=messages,
                )
                return resp.choices[0].message.content, provider["name"]
        except Exception as exc:
            last_exc = exc
            if _is_quota_error(exc):
                print(f"  [{provider['name']}] quota/rate-limit hit — trying next ...")
                continue
            else:
                raise

    raise RuntimeError(f"All providers exhausted. Last error: {last_exc}")


print("Direct LLM function ready (for without-RAG comparison).")

Direct LLM function ready (for without-RAG comparison).


## 7) Run End-to-End Examples: With RAG vs. Without RAG

We test questions about Acme Analytics — a **fictional company** that no LLM has seen in training.
This makes the with/without RAG comparison very clear:
- With RAG: correct, specific answers grounded on our documents
- Without RAG: refusals, hedging, or hallucinated numbers

In [8]:
questions = [
    "What was Acme Analytics' revenue in Q4 2025?",
    #"How many days of PTO do new employees get at Acme?",
    #"What products does Acme Analytics offer and what do they cost?",
    #"How much did BigRetail Corp save using Acme's products?",
    "Who founded Acme Analytics and where is the company headquartered?",
]


def preview(text, max_len=800):
    text = str(text) if text else ""
    return text[:max_len] + ("..." if len(text) > max_len else "")


if not has_llm_provider():
    print("Error: Set at least one API key before running.")
else:
    for i, q in enumerate(questions, start=1):
        print("\n" + "=" * 80)
        print(f"Q{i}. {q}")
        print("=" * 80)

        # --- WITH RAG (LlamaIndex) ---
        print("\n--- WITH RAG (LlamaIndex) ---")
        try:
            response = query_engine.query(q)
            print(f"  Answer:")
            print(preview(response.response))
            # Show which sources were used
            sources = set()
            for node in response.source_nodes:
                sources.add(node.metadata.get('file_name', 'unknown'))
            print(f"  Sources: {', '.join(sources)}")
        except Exception as e:
            print(f"  RAG error: {e}")

        # --- WITHOUT RAG ---
        print("\n--- WITHOUT RAG (LLM knowledge only) ---")
        try:
            answer_direct, prov_direct = generate_text_no_rag(q)
            print(f"  Answer (provider: {prov_direct}):")
            print(preview(answer_direct))
        except Exception as e:
            print(f"  Direct error: {e}")

        print()


Q1. What was Acme Analytics' revenue in Q4 2025?

--- WITH RAG (LlamaIndex) ---
  Answer:
Acme Analytics' revenue in Q4 2025 was $12.8 million.
  Sources: products_and_services.txt, q4_2025_financials.txt, company_overview.txt

--- WITHOUT RAG (LLM knowledge only) ---
  Answer (provider: Gemini):
I'm sorry, but as an AI, I do not have access to real-time, future, or specific private financial data for companies like Acme Analytics.

My knowledge cutoff is **June 2024**, and I cannot predict future financial performance or access proprietary information that hasn't been publicly released.

To find out Acme Analytics' revenue in Q4 2025, you would need to:
*   Wait until the company publicly releases its Q4 2025 financial report.
*   Consult financial news sources or the company's official investor relations if it's a publicly traded company.
*   Have access to their internal financial records if you are an authorized party.


Q2. Who founded Acme Analytics and where is the company head

### Checkpoint: Reflection Questions

1. **Hallucination check:** Did the without-RAG answers invent specific revenue numbers or employee counts for a fictional company?
2. **Source attribution:** LlamaIndex tells us *which document* the answer came from. Why is this important for trust?
3. **Chunking:** How does the chunk size affect retrieval quality? What if a key fact spans two chunks?
4. **Framework vs. manual:** Compare this LlamaIndex approach to the manual RAG we built in the VectorDB and DBMS_RAG notebooks. What did the framework handle for us?

## 8) Interactive Query (Optional)

Ask your own questions about Acme Analytics.

In [9]:
# Change this to any question about Acme Analytics
my_question = "What is Acme's LTV/CAC ratio and what does it mean?"

if has_llm_provider():
    print(f"Question: {my_question}\n")

    print("=" * 60)
    print("WITH RAG (LlamaIndex):")
    print("=" * 60)
    try:
        response = query_engine.query(my_question)
        print(f"\n{response.response}")
        sources = set()
        for node in response.source_nodes:
            sources.add(node.metadata.get('file_name', 'unknown'))
        print(f"\nSources: {', '.join(sources)}")
    except Exception as e:
        print(f"Error: {e}")

    print("\n" + "=" * 60)
    print("WITHOUT RAG:")
    print("=" * 60)
    try:
        answer_direct, prov = generate_text_no_rag(my_question)
        print(f"\n[{prov}] {answer_direct}")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Set at least one API key first.")

Question: What is Acme's LTV/CAC ratio and what does it mean?

WITH RAG (LlamaIndex):

Acme's LTV/CAC ratio is 8.2x.

This ratio indicates that the lifetime value of a customer is 8.2 times greater than the cost to acquire that customer. A higher LTV/CAC ratio generally suggests that the company is efficiently acquiring customers who generate significant revenue over their relationship with the business, making customer acquisition a profitable endeavor.

Sources: q4_2025_financials.txt, employee_handbook_excerpt.txt, company_overview.txt

WITHOUT RAG:

[Gemini] Since I don't have real-time financial data for a specific company named "Acme," I'll provide a **hypothetical example** to explain Acme's LTV/CAC ratio and its meaning.

---

### Hypothetical Example for Acme:

Let's assume Acme's key metrics are:

*   **Average Customer Lifetime Value (LTV):** $1,000
*   **Average Customer Acquisition Cost (CAC):** $200

**Acme's LTV/CAC Ratio:**

LTV / CAC = $1,000 / $200 = **5:1**

---

###

## 9) Bonus: Customizing the RAG Pipeline

LlamaIndex is highly configurable. Here we show how to tweak:
- **Chunk size** — smaller chunks = more precise retrieval, larger = more context per chunk
- **Top-k** — how many chunks to retrieve
- **System prompt** — customize the LLM's behavior

In [10]:
from llama_index.core.node_parser import SentenceSplitter

# Re-index with smaller chunks and more overlap
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=50)
small_chunk_index = VectorStoreIndex.from_documents(
    documents,
    transformations=[splitter],
    show_progress=True,
)

# Query with more chunks retrieved
custom_engine = small_chunk_index.as_query_engine(similarity_top_k=5)

test_q = "What is the pricing for InsightBoard Pro?"
print(f"Question: {test_q}\n")

if has_llm_provider():
    try:
        response = custom_engine.query(test_q)
        print(f"Answer (small chunks, top_k=5):")
        print(response.response)
        print(f"\nChunks used: {len(response.source_nodes)}")
        for i, node in enumerate(response.source_nodes, 1):
            print(f"  #{i} score={node.score:.4f} | {node.metadata.get('file_name', '?')}")
            print(f"     {node.text[:100].replace(chr(10), ' ')}...")
    except Exception as e:
        print(f"Error: {e}")

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

Question: What is the pricing for InsightBoard Pro?



Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 33.718687279s.


## 10) Teaching Notes and Exercises

**Key takeaways:**
- **LlamaIndex** abstracts the RAG pipeline: document loading, chunking, embedding, indexing, retrieval, and LLM prompting — all in a few lines of code.
- Under the hood, it does the **same thing** as our manual RAG notebooks (embed → retrieve → generate), but with less boilerplate.
- **Source attribution** (which document chunk answered the question) comes for free.
- The framework is **highly configurable**: chunk size, overlap, top-k, embedding model, LLM, and prompt templates can all be swapped.
- Using **synthetic/private data** makes the with-vs-without-RAG comparison very clear — the LLM literally cannot answer without retrieval.

**LlamaIndex vs. Manual RAG — When to use which:**

| Aspect | Manual RAG | LlamaIndex |
|---|---|---|
| Learning | Better for understanding fundamentals | Hides implementation details |
| Speed to prototype | Slower | Much faster |
| Customization | Full control | Configurable, but within framework |
| Production readiness | Build it yourself | Batteries included |

**Exercises:**
- Add more documents (e.g., a product roadmap, a competitor analysis) and see how retrieval adapts.
- Experiment with different chunk sizes (128, 512, 1024) and compare answer quality.
- Try using a different embedding model (e.g., `BAAI/bge-small-en-v1.5`) and compare retrieval accuracy.
- Add a **metadata filter** so queries only search specific document types (e.g., only financials).
- Discuss: In a real business setting, what documents would you index? What are the security/privacy implications?